# Video Category — Silver Transformation

- Read the Bronze video category data
- Explode the nested category records
- Select and flatten the required fields
- Rename and cast columns to the Silver schema
- Write the transformed data to a Silver Delta table
- Validate row count and category ID uniqueness

## 1. Inspect Bronze data

In [0]:
%sql
SELECT *
FROM youtube_content_intelligence.bronze.brz_video_category;

In [0]:
%sql
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_video_category;

## 2. Transform Bronze data

In [0]:
df_bronze = spark.table("youtube_content_intelligence.bronze.brz_video_category")

### Explode nested category records

In [0]:
from pyspark.sql.functions import explode

df_exploded = df_bronze.select(explode("items").alias("item"))

In [0]:
display(df_exploded.limit(5))

### Select required fields

In [0]:
df_selected = df_exploded.select(
    "item.id",
    "item.snippet.title"
)

In [0]:
display(df_selected.limit(5))

### Rename and cast columns

In [0]:
from pyspark.sql.functions import col

df_silver = df_selected.select(
    col("id").cast("int").alias("category_id"),
    col("title").alias("category")
)

In [0]:
df_silver.printSchema()

In [0]:
display(df_silver.limit(5))

## 3. Write to Silver

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(
    "youtube_content_intelligence.silver.slv_video_category"
)

## 4. Validate Silver table

In [0]:
%sql
-- Preview Silver data
SELECT *
FROM youtube_content_intelligence.silver.slv_video_category
LIMIT 5;

In [0]:
%sql
-- Check row count
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.silver.slv_video_category;

In [0]:
%sql
-- Check duplicate category IDs
SELECT category_id, COUNT(*) AS count
FROM youtube_content_intelligence.silver.slv_video_category
GROUP BY category_id
HAVING COUNT(*) > 1;